In [ ]:
import sys

sys.path.append("..")

from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import cartopy.crs as crs
import cartopy.feature as cfeature
from shapely.geometry import Point
import shapely.vectorized as sv

from src.data import nysm_data

In [ ]:
def date_filter(ldf, time1, time2):
    ldf = ldf[ldf["valid_time"] > time1]
    ldf = ldf[ldf["valid_time"] < time2]

    return ldf


def idw_interpolation(x, y, z, xi, yi, power=2):
    """
    x, y = known points
    z = values
    xi, yi = meshgrid of where to interpolate
    """
    dist = np.sqrt((xi[..., None] - x) ** 2 + (yi[..., None] - y) ** 2)

    # Avoid division by zero
    dist = np.where(dist == 0, 1e-12, dist)

    weights = 1 / dist**power
    z_idw = np.sum(weights * z, axis=-1) / np.sum(weights, axis=-1)
    return z_idw


def plot_nysm(nysm_df, lons, lats, values, var):
    df_ = nysm_df.copy()
    font_size = 22

    fig = plt.figure(figsize=(24, 16))
    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=crs.LambertConformal(
            central_longitude=-75.0, standard_parallels=(49, 77)
        ),
    )

    # Load shapefile
    ny_state_boundaries_path = "/home/aevans/nwp_bias/src/landtype/data/State.shx"
    ny_state_boundaries_geo = gpd.read_file(ny_state_boundaries_path).to_crs(epsg=4326)

    # Get bounds for the map
    min_lon = nysm_df["lon"].min()
    max_lon = nysm_df["lon"].max()
    min_lat = nysm_df["lat"].min()
    max_lat = nysm_df["lat"].max()
    pad = 0.1

    extent = [min_lon - pad, max_lon + pad, min_lat - pad, max_lat + pad]
    ax.set_extent(extent, crs=crs.PlateCarree())

    # Features
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), zorder=1)

    # Gridlines
    ax.gridlines(
        crs=crs.PlateCarree(),
        draw_labels=True,
        linewidth=2,
        color="black",
        alpha=0.5,
        linestyle="--",
    )

    grid_res = 500
    grid_lon, grid_lat = np.meshgrid(
        np.linspace(min_lon - pad, max_lon + pad, grid_res),
        np.linspace(min_lat - pad, max_lat + pad, grid_res),
    )

    # IDW interpolation
    zi = idw_interpolation(x=lons, y=lats, z=values, xi=grid_lon, yi=grid_lat)

    # Mask outside NY
    ny_poly = ny_state_boundaries_geo.unary_union
    mask = ~sv.contains(ny_poly, grid_lon, grid_lat)
    zi_masked = np.ma.array(zi, mask=mask)

    # Interpolated field
    c = ax.pcolormesh(
        grid_lon,
        grid_lat,
        zi_masked,
        cmap="Greens",
        transform=crs.PlateCarree(),
        shading="auto",
    )

    cbar = plt.colorbar(c, ax=ax, shrink=0.6)
    cbar.set_label("Max Wind (m s)", fontsize=font_size)
    cbar.ax.tick_params(labelsize=font_size)

    # Station scatter
    sc = ax.scatter(
        df_["lon"],
        df_["lat"],
        s=100,
        c="black",
        edgecolor="black",
        transform=crs.PlateCarree(),
        zorder=10,
    )

    # Annotate stations
    for _, row in df_.iterrows():
        ax.annotate(
            row["station"],
            (row["lon"], row["lat"]),
            textcoords="offset points",
            xytext=(0, 15),
            ha="center",
            fontsize=15,
            color="black",
            transform=crs.PlateCarree(),
            zorder=20,
        )

    # Boundary overlay
    ny_state_boundaries_geo.boundary.plot(ax=ax, edgecolor="black", linewidth=2)

    # Legend and title
    plt.title(f"NYSM: Wind Storm", fontsize=font_size)

    plt.tight_layout()
    plt.show()
    plt.savefig(
        "/home/aevans/nwp_bias/src/machine_learning/data/error_visuals/ALL/precip_high_impact.png"
    )

In [ ]:
def main(stations, time1, time2, var, method):
    # load nysm data
    nysm_df = nysm_data.load_nysm_data(gfs=False)
    nysm_df = nysm_df.rename(columns={"time_1H": "valid_time"})
    print(nysm_df.columns)

    # filter for effected stations
    nysm_df = nysm_df[nysm_df["station"].isin(stations)]

    # filter for time
    nysm_df = date_filter(nysm_df, time1, time2)

    if method == "accumulate":
        print("accumulating")
        nysm_df = nysm_df.sort_values(["station", "valid_time"])
        nysm_df[f"{var}_t"] = nysm_df.groupby("station")[var].cumsum()

    nysm_df.dropna(inplace=True)
    values = []
    lats = []
    lons = []

    for s in nysm_df["station"].unique():
        df_ = nysm_df[nysm_df["station"] == s]

        # iloc[-1] returns a scalar, so NO .values
        # values.append(df_[f"{var}_t"].iloc[-1])
        values.append(df_[f"{var}"].max())
        lats.append(df_["lat"].iloc[-1])
        lons.append(df_["lon"].iloc[-1])

    # Convert lists → arrays
    values = np.array(values, dtype=float)
    lats = np.array(lats, dtype=float)
    lons = np.array(lons, dtype=float)
    plot_nysm(nysm_df, lons, lats, values, var)

In [ ]:
time1 = datetime(2024, 8, 8, 0, 0, 0)
time2 = datetime(2024, 8, 11, 23, 59, 59)

nysm_clim = pd.read_csv("/home/aevans/nwp_bias/src/landtype/data/nysm.csv")
# stations = nysm_clim['stid'].unique()

# c = 'Champlain Valley'

use_ls = ["Champlain Valley", "Northern Plateau"]
nysm_ = nysm_clim[nysm_clim["climate_division_name"].isin(use_ls)]

# nysm_ = nysm_clim[nysm_clim['climate_division_name']==c]
stations = nysm_["stid"].unique()

main(stations, time1, time2, "wmax_sonic", "max")